In [1]:
import os
import numpy as np
import pandas as pd
import random
import librosa

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import AutoFeatureExtractor, ASTForAudioClassification

In [2]:
GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]

recipes = []

for genre in GENRES:
    for i in range(100):
        recipes.append({
            "genre": genre,
            "id": i
        })

train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Validation size:", len(val_recipes))  # 200

Validation size: 200


In [3]:
def create_dummy_mix():
    stems = []

    for _ in range(4):
        y = np.random.randn(160000)
        stems.append(y)

    noise = np.random.randn(160000)

    mix = sum(stems)
    mix = mix + 0.2 * noise

    return mix

mix = create_dummy_mix()
print(mix.shape)  # (160000,)

(160000,)


In [4]:
extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

mix = np.ones(160000)

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

features = inputs["input_values"].squeeze(0)

print(features.shape)  # (1024, 128)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

torch.Size([1024, 128])


In [5]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable params:", params)  # 87406318

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable params: 86196490


In [6]:
class AudioDataset(Dataset):
    def __init__(self, recipes, extractor):
        self.recipes = recipes
        self.extractor = extractor

    def __len__(self):
        return len(self.recipes)

    def __getitem__(self, idx):
        mix = create_dummy_mix()

        # normalization
        mix = mix / (np.max(np.abs(mix)) + 1e-9)

        inputs = self.extractor(
            mix,
            sampling_rate=16000,
            return_tensors="pt"
        )

        x = inputs["input_values"].squeeze(0)

        label = random.randint(0, 9)

        return x, label

In [7]:
train_ds = AudioDataset(train_recipes, extractor)
val_ds = AudioDataset(val_recipes, extractor)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        outputs = model(input_values=x)
        logits = outputs.logits

        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

NameError: name 'device' is not defined

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs.logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss:", total_loss / len(train_loader))

In [ ]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print(round(y_norm[0], 3))  # -1.000

In [ ]:
def predict(audio):
    audio = audio / (np.max(np.abs(audio)) + 1e-9)

    inputs = extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    x = inputs["input_values"].to(device)

    with torch.no_grad():
        logits = model(x).logits

    return logits.argmax(dim=1).item()